# Retrieval Pipeline Evaluation

Notebook for offline evaluation of the baseline retrievers, category classification, and classifier-enhanced retrieval variants. All 327 labeled training queries are treated as the canonical offline evaluation set.

In [1]:
import sys
from pathlib import Path

def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return 'colab'
    except Exception:
        if Path('/kaggle/input').exists():
            return 'kaggle'
        return 'local'

def add_project_root_to_syspath(project_name: str = 'retrieval_project') -> None:
    runtime_env = detect_runtime_environment()
    candidates = [Path.cwd(), *Path.cwd().parents]
    if runtime_env == 'colab':
        drive_root = Path('/content/drive/MyDrive')
        if not drive_root.exists():
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        candidates = [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), *candidates]
    elif runtime_env == 'kaggle':
        candidates = [Path('/kaggle/working'), *candidates]

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base))
            return
        if (base / project_name / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base / project_name))
            return
        if runtime_env == 'colab' and base.exists():
            for match in base.rglob(project_name):
                if (match / 'src' / 'infra' / 'notebook.py').exists():
                    sys.path.insert(0, str(match))
                    return
    raise FileNotFoundError('Could not locate project root containing src/infra/notebook.py')

add_project_root_to_syspath()

from src.infra.notebook import setup_notebook

runtime_env, project_root = setup_notebook()
print(f'Detected runtime: {runtime_env}')
print(f'Project root    : {project_root}')


Detected runtime: local
Project root    : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project


## Step 1: Load Data and Build a Notebook-Local Evaluation Config

In [2]:
from dataclasses import replace

import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from src.categorization import build_or_load_category_classifier, predict_category_map
from src.config import DEFAULT_CONFIG
from src.evaluation import leaderboard_score, load_ground_truth
from src.pipeline import (
    bootstrap,
    build_cross_encoder_reranker,
    load_project_frames,
    predict_categories,
    prepare_retrievers,
    run_first_stage_retrieval,
)
from src.reranking import rerank_results_with_cross_encoder
from src.retrieval import run_retrieval_scored, truncate_results

paths, runtime_config = bootstrap()
evaluation_config = replace(
    runtime_config,
    retrieval_pipeline=replace(
        runtime_config.retrieval_pipeline,
        final_model='embedding',
        evaluation_models=('tfidf', 'bm25', 'embedding'),
        evaluation_top_ks=(10, 100, 1000, 7500, 12500),
        submit_top_k=12500,
        enable_category_prediction=True,
        enable_category_filter=True,
        enable_cross_encoder_rerank=True,
    ),
    cross_encoder=replace(
        runtime_config.cross_encoder,
        rerank_top_m=45,
        category_bonus=0.5,
        max_length=256,
    ),
)
frames = load_project_frames(paths, evaluation_config)
ground_truth = load_ground_truth(paths.data_dir / 'qgts_train.json')
phase1_top_ks = evaluation_config.retrieval_pipeline.evaluation_top_ks
phase1_max_top_k = max(phase1_top_ks)
phase2_top_k = 12500

print(f'Documents                  : {len(frames.docs):,}')
print(f'Labeled offline queries    : {len(frames.train_queries):,}')
print(f'Kaggle test queries        : {len(frames.test_queries):,}')
print(f'Offline evaluation models  : {evaluation_config.retrieval_pipeline.evaluation_models}')
print(f'Phase 1 K grid             : {phase1_top_ks}')
print(f'Phase 2 TopK               : {phase2_top_k:,}')
print(f'Rerank top_m               : {evaluation_config.cross_encoder.rerank_top_m}')
print(f'Category bonus             : {evaluation_config.cross_encoder.category_bonus:.2f}')
print(f'Classifier query tags used : {evaluation_config.data_columns.use_query_tags_in_classifier}')


Documents                  : 216,041
Labeled offline queries    : 327
Kaggle test queries        : 141
Offline evaluation models  : ('tfidf', 'bm25', 'embedding')
Phase 1 K grid             : (10, 100, 1000, 7500, 12500)
Phase 2 TopK               : 12,500
Rerank top_m               : 45
Category bonus             : 0.50
Classifier query tags used : True


## Step 2: Prepare Shared Artifacts

In [3]:
prepared_retrievers = prepare_retrievers(frames, paths, config=evaluation_config)
category_artifacts = predict_categories(frames, paths, ground_truth=ground_truth, config=evaluation_config)
cross_encoder_reranker = build_cross_encoder_reranker(frames, paths, ground_truth, config=evaluation_config)

print(f'Prepared retrievers: {sorted(prepared_retrievers.keys())}')
print(f'Category accuracy  : {category_artifacts.classifier_accuracy:.5f}')
print(f'Reranker ready     : {cross_encoder_reranker is not None}')


Loading BM25 index from cache: bm25_d750cb844bef214c.pkl


Loading model weights from cache: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/cache/sentence_transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_6485efc872cab480.npy


Loading TF-IDF artifacts from cache: tfidf_63f6d61f0762518b.pkl


Loading category classifier from cache: category_classifier_e3f41fe8a612f7c0.pkl


Loading cross-encoder from cache: cross-encoder_ms-marco-MiniLM-L6-v2_c2a686d8799ae524


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Prepared retrievers: ['bm25', 'embedding', 'tfidf']
Category accuracy  : 0.92661
Reranker ready     : True


## Step 3: Phase 1 Baseline Retriever Comparison

Each baseline model is run once at the maximum K. Smaller-K metrics are then derived by truncation so the notebook exposes both the explicit `topk_indices_*` / `topk_scores_*` arrays and the K-sensitivity tables required by the assignment.

In [4]:
baseline_outputs_by_model = {}
for model_name in evaluation_config.retrieval_pipeline.evaluation_models:
    print(f'Running scored baseline retrieval for: {model_name}')
    baseline_outputs_by_model[model_name] = run_retrieval_scored(
        model_name=model_name,
        docs_frame=frames.docs,
        queries_frame=frames.train_queries,
        top_k=phase1_max_top_k,
        cache_dir=paths.cache_dir,
        prepared_artifacts=prepared_retrievers[model_name],
        embedding_kind='queries_train_phase1',
        config=evaluation_config,
    )

topk_indices_tfidf = baseline_outputs_by_model['tfidf'].topk_indices
topk_scores_tfidf = baseline_outputs_by_model['tfidf'].topk_scores
topk_indices_bm25 = baseline_outputs_by_model['bm25'].topk_indices
topk_scores_bm25 = baseline_outputs_by_model['bm25'].topk_scores
topk_indices_embedding = baseline_outputs_by_model['embedding'].topk_indices
topk_scores_embedding = baseline_outputs_by_model['embedding'].topk_scores

for model_name, output in baseline_outputs_by_model.items():
    latency_ms = 1000.0 * output.elapsed_seconds / max(len(output.results), 1)
    print(f"{model_name:>9} -> elapsed={output.elapsed_seconds:.2f}s, latency/query={latency_ms:.2f} ms")
print(f'topk_indices_tfidf shape     : {topk_indices_tfidf.shape}')
print(f'topk_scores_tfidf shape      : {topk_scores_tfidf.shape}')
print(f'topk_indices_bm25 shape      : {topk_indices_bm25.shape}')
print(f'topk_scores_bm25 shape       : {topk_scores_bm25.shape}')
print(f'topk_indices_embedding shape : {topk_indices_embedding.shape}')
print(f'topk_scores_embedding shape  : {topk_scores_embedding.shape}')


Running scored baseline retrieval for: tfidf
Starting retrieval: model=tfidf
  parameters: top_k=12,500, docs=216,041, queries=327, prepared_artifacts=yes, embedding_kind='queries_train_phase1'
  [TF-IDF] vectorized 327 queries against 216,041 docs with capped_top_k=12,500


  [TF-IDF] processed 65/327 queries


  [TF-IDF] processed 130/327 queries


  [TF-IDF] processed 195/327 queries


  [TF-IDF] processed 260/327 queries


...

  [Embedding] chunk 3/11: queries 65-96
  [Embedding] chunk 4/11: queries 97-128


  [Embedding] chunk 5/11: queries 129-160
  [Embedding] chunk 6/11: queries 161-192


  [Embedding] chunk 7/11: queries 193-224
  [Embedding] chunk 8/11: queries 225-256


  [Embedding] chunk 9/11: queries 257-288
  [Embedding] chunk 10/11: queries 289-320


  [Embedding] chunk 11/11: queries 321-327
Completed retrieval: model=embedding, results=327 queries, elapsed=1.2s
    tfidf -> elapsed=119.47s, latency/query=365.34 ms
     bm25 -> elapsed=143.12s, latency/query=437.68 ms
embedding -> elapsed=1.20s, latency/query=3.68 ms
topk_indices_tfidf shape     : (327, 12500)
topk_scores_tfidf shape      : (327, 12500)
topk_indices_bm25 shape      : (327, 12500)
topk_scores_bm25 shape       : (327, 12500)
topk_indices_embedding shape : (327, 12500)
topk_scores_embedding shape  : (327, 12500)


In [5]:
phase1_rows = []
for model_name, output in baseline_outputs_by_model.items():
    latency_ms = 1000.0 * output.elapsed_seconds / max(len(output.results), 1)
    for top_k in phase1_top_ks:
        metrics = leaderboard_score(
            truncate_results(output.results, top_k),
            ground_truth,
            k=top_k,
            accuracy_value=category_artifacts.classifier_accuracy,
        )
        phase1_rows.append(
            {
                'Model': model_name,
                'TopK': int(top_k),
                'ElapsedSeconds': output.elapsed_seconds,
                'LatencyPerQueryMs': latency_ms,
                **metrics,
            }
        )

phase1_summary_df = pd.DataFrame(phase1_rows).sort_values(['Model', 'TopK']).reset_index(drop=True)
display(phase1_summary_df)
phase1_best_by_model_df = (
    phase1_summary_df.sort_values(['LeaderboardScore', 'MRR', 'Recall', 'TopK'], ascending=[False, False, False, False])
    .groupby('Model', as_index=False)
    .head(1)
    .sort_values('Model')
    .reset_index(drop=True)
)
display(phase1_best_by_model_df)


,Model,TopK,ElapsedSeconds,LatencyPerQueryMs,Recall,Precision,MRR,Accuracy,LeaderboardScore
0,bm25,10,143.122258,437.682745,0.151892,0.092355,0.271822,0.926606,0.360668
1,bm25,100,143.122258,437.682745,0.318234,0.020795,0.281283,0.926606,0.386729
2,bm25,1000,143.122258,437.682745,0.510815,0.003734,0.281970,0.926606,0.430781
3,bm25,7500,143.122258,437.682745,0.681980,0.000702,0.282020,0.926606,0.472827
4,bm25,12500,143.122258,437.682745,0.717214,0.000449,0.282023,0.926606,0.481573
5,embedding,10,1.204955,3.684877,0.311335,0.183486,0.466540,0.926606,0.471992
6,embedding,100,1.204955,3.684877,0.581491,0.040489,0.474049,0.926606,0.505659
7,embedding,1000,1.204955,3.684877,0.828389,0.006431,0.474495,0.926606,0.558980
8,embedding,7500,1.204955,3.684877,0.947617,0.001099,0.474507,0.926606,0.587457
9,embedding,12500,1.204955,3.684877,0.965746,0.000685,0.474507,0.926606,0.591886


,Model,TopK,ElapsedSeconds,LatencyPerQueryMs,Recall,Precision,MRR,Accuracy,LeaderboardScore
0,bm25,12500,143.122258,437.682745,0.717214,0.000449,0.282023,0.926606,0.481573
1,embedding,12500,1.204955,3.684877,0.965746,0.000685,0.474507,0.926606,0.591886
2,tfidf,12500,119.467450,365.343883,0.790170,0.000490,0.262707,0.926606,0.494993


## Step 4: Category Classification

Accuracy is computed as the number of correctly predicted categories divided by the total number of evaluated examples.

In [6]:
docs_classifier_df = frames.docs_classifier[['id', 'content', 'category']].copy()
docs_classifier_df['category'] = docs_classifier_df['category'].astype(str)
doc_train_df, doc_holdout_df = train_test_split(
    docs_classifier_df,
    test_size=0.2,
    random_state=42,
    stratify=docs_classifier_df['category'],
)

holdout_classifier_artifacts = build_or_load_category_classifier(
    train_frame=doc_train_df,
    cache_dir=paths.cache_dir,
    config=evaluation_config,
)
doc_holdout_pred_map = predict_category_map(doc_holdout_df[['id', 'content']], holdout_classifier_artifacts)
doc_holdout_eval_df = doc_holdout_df[['id', 'category']].copy()
doc_holdout_eval_df['predicted_category'] = doc_holdout_eval_df['id'].astype(str).map(doc_holdout_pred_map)
doc_holdout_accuracy = accuracy_score(doc_holdout_eval_df['category'], doc_holdout_eval_df['predicted_category'])

labels = sorted(doc_holdout_eval_df['category'].unique())
doc_classification_report_df = pd.DataFrame(
    classification_report(
        doc_holdout_eval_df['category'],
        doc_holdout_eval_df['predicted_category'],
        labels=labels,
        output_dict=True,
        zero_division=0,
    )
).T
doc_confusion_df = pd.DataFrame(
    confusion_matrix(doc_holdout_eval_df['category'], doc_holdout_eval_df['predicted_category'], labels=labels),
    index=labels,
    columns=labels,
)
doc_per_category_accuracy_df = (
    doc_holdout_eval_df.assign(correct=lambda frame: frame['category'] == frame['predicted_category'])
    .groupby('category', as_index=False)['correct']
    .mean()
    .rename(columns={'category': 'Category', 'correct': 'Accuracy'})
    .sort_values('Category')
    .reset_index(drop=True)
)

print(f'Document holdout accuracy: {doc_holdout_accuracy:.5f}')
display(doc_per_category_accuracy_df)
display(doc_classification_report_df)
display(doc_confusion_df)


Saved category classifier to cache: category_classifier_c911cc29cbfcca1c.pkl


Document holdout accuracy: 0.98604


,Category,Accuracy
0,android,0.975652
1,gaming,0.994481
2,programmers,0.974204
3,tex,0.993987
4,unix,0.979635


,precision,recall,f1-score,support
android,0.985291,0.975652,0.980448,4600.000000
gaming,0.993385,0.994481,0.993933,9060.000000
programmers,0.977089,0.974204,0.975644,6435.000000
tex,0.994060,0.993987,0.994023,13637.000000
unix,0.973982,0.979635,0.976800,9477.000000
accuracy,0.986045,0.986045,0.986045,0.986045
macro avg,0.984761,0.983592,0.984170,43209.000000
weighted avg,0.986054,0.986045,0.986044,43209.000000


,android,gaming,programmers,tex,unix
android,4488,19,15,3,75
gaming,15,9010,11,0,24
programmers,23,23,6269,25,95
tex,1,4,23,13555,54
unix,28,14,98,53,9284


In [7]:
train_query_eval_df = frames.train_queries_classifier[['id', 'content', 'category']].copy()
train_query_eval_df['category'] = train_query_eval_df['category'].astype(str)
train_query_pred_map = predict_category_map(train_query_eval_df[['id', 'content']], holdout_classifier_artifacts)
train_query_eval_df['predicted_category'] = train_query_eval_df['id'].astype(str).map(train_query_pred_map)
train_query_accuracy = accuracy_score(train_query_eval_df['category'], train_query_eval_df['predicted_category'])

document_vs_query_df = pd.DataFrame(
    [
        {'EvaluationSet': 'document_holdout', 'Accuracy': doc_holdout_accuracy, 'Samples': len(doc_holdout_eval_df)},
        {'EvaluationSet': 'train_queries_diagnostic', 'Accuracy': train_query_accuracy, 'Samples': len(train_query_eval_df)},
    ]
)
print(f'Train-query diagnostic accuracy: {train_query_accuracy:.5f}')
display(document_vs_query_df)


Train-query diagnostic accuracy: 0.93272


,EvaluationSet,Accuracy,Samples
0,document_holdout,0.986045,43209
1,train_queries_diagnostic,0.932722,327


## Step 5: Phase 2 Classifier-Enhanced Retrieval Comparison

In [8]:
query_category_map = category_artifacts.train_query_category_map or {}
doc_category_map = category_artifacts.doc_category_map or {}
if cross_encoder_reranker is None:
    raise ValueError('Cross-encoder reranking is required for the Phase 2 comparison.')

phase2_rows = []
phase2_variant_tables = {}
phase2_best_by_model = []
variant_order = [
    'baseline',
    'category_filtered',
    'reranked',
    'reranked_plus_category_bonus',
    'category_filtered_plus_rerank',
]


def build_phase2_config(model_name: str, enable_category_filter: bool, enable_cross_encoder_rerank: bool):
    return replace(
        evaluation_config,
        retrieval_pipeline=replace(
            evaluation_config.retrieval_pipeline,
            final_model=model_name,
            submit_top_k=phase2_top_k,
            evaluation_top_ks=(phase2_top_k,),
            enable_category_filter=enable_category_filter,
            enable_cross_encoder_rerank=enable_cross_encoder_rerank,
        ),
    )


for model_name in evaluation_config.retrieval_pipeline.evaluation_models:
    print()
    print(f'===== Phase 2 comparison for {model_name} =====')
    base_config = build_phase2_config(model_name, enable_category_filter=False, enable_cross_encoder_rerank=False)
    filtered_config = build_phase2_config(model_name, enable_category_filter=True, enable_cross_encoder_rerank=False)

    baseline_results, _ = run_first_stage_retrieval(
        frames=frames,
        paths=paths,
        prepared_retrievers=prepared_retrievers,
        category_artifacts=category_artifacts,
        split='train',
        top_k=phase2_top_k,
        model_name_override=model_name,
        config=base_config,
    )
    category_filtered_results, _ = run_first_stage_retrieval(
        frames=frames,
        paths=paths,
        prepared_retrievers=prepared_retrievers,
        category_artifacts=category_artifacts,
        split='train',
        top_k=phase2_top_k,
        model_name_override=model_name,
        config=filtered_config,
    )

    reranked_results = rerank_results_with_cross_encoder(
        results=baseline_results,
        query_frame=frames.train_queries,
        docs_frame=frames.docs,
        cross_encoder=cross_encoder_reranker,
        query_category_map=query_category_map,
        doc_category_map=doc_category_map,
        infer_batch_size=evaluation_config.cross_encoder.infer_batch_size,
        rerank_top_m=evaluation_config.cross_encoder.rerank_top_m,
        category_bonus=0.0,
    )
    reranked_bonus_results = rerank_results_with_cross_encoder(
        results=baseline_results,
        query_frame=frames.train_queries,
        docs_frame=frames.docs,
        cross_encoder=cross_encoder_reranker,
        query_category_map=query_category_map,
        doc_category_map=doc_category_map,
        infer_batch_size=evaluation_config.cross_encoder.infer_batch_size,
        rerank_top_m=evaluation_config.cross_encoder.rerank_top_m,
        category_bonus=evaluation_config.cross_encoder.category_bonus,
    )
    filtered_reranked_results = rerank_results_with_cross_encoder(
        results=category_filtered_results,
        query_frame=frames.train_queries,
        docs_frame=frames.docs,
        cross_encoder=cross_encoder_reranker,
        query_category_map=query_category_map,
        doc_category_map=doc_category_map,
        infer_batch_size=evaluation_config.cross_encoder.infer_batch_size,
        rerank_top_m=evaluation_config.cross_encoder.rerank_top_m,
        category_bonus=evaluation_config.cross_encoder.category_bonus,
    )

    variant_results = {
        'baseline': baseline_results,
        'category_filtered': category_filtered_results,
        'reranked': reranked_results,
        'reranked_plus_category_bonus': reranked_bonus_results,
        'category_filtered_plus_rerank': filtered_reranked_results,
    }

    model_rows = []
    for variant_name in variant_order:
        metrics = leaderboard_score(
            truncate_results(variant_results[variant_name], phase2_top_k),
            ground_truth,
            k=phase2_top_k,
            accuracy_value=category_artifacts.classifier_accuracy,
        )
        row = {'Model': model_name, 'Variant': variant_name, 'TopK': phase2_top_k, **metrics}
        phase2_rows.append(row)
        model_rows.append(row)

    model_df = pd.DataFrame(model_rows)
    baseline_score = float(model_df.loc[model_df['Variant'] == 'baseline', 'LeaderboardScore'].iloc[0])
    model_df['DeltaVsBaseline'] = model_df['LeaderboardScore'] - baseline_score
    model_df['Variant'] = pd.Categorical(model_df['Variant'], categories=variant_order, ordered=True)
    model_df = model_df.sort_values('Variant').reset_index(drop=True)
    phase2_variant_tables[model_name] = model_df

    best_row = model_df.sort_values(['LeaderboardScore', 'MRR', 'Recall'], ascending=[False, False, False]).iloc[0].to_dict()
    best_row['Variant'] = str(best_row['Variant'])
    phase2_best_by_model.append(best_row)

phase2_summary_df = pd.DataFrame(phase2_rows)
baseline_lookup = (
    phase2_summary_df[phase2_summary_df['Variant'] == 'baseline'][['Model', 'LeaderboardScore']]
    .rename(columns={'LeaderboardScore': 'BaselineLeaderboardScore'})
)
phase2_summary_df = phase2_summary_df.merge(baseline_lookup, on='Model', how='left')
phase2_summary_df['DeltaVsBaseline'] = phase2_summary_df['LeaderboardScore'] - phase2_summary_df['BaselineLeaderboardScore']
phase2_summary_df['Variant'] = pd.Categorical(phase2_summary_df['Variant'], categories=variant_order, ordered=True)
phase2_summary_df = phase2_summary_df.sort_values(['Model', 'Variant']).reset_index(drop=True)

for model_name in evaluation_config.retrieval_pipeline.evaluation_models:
    display(phase2_variant_tables[model_name])

phase2_cross_model_pivot = phase2_summary_df.pivot_table(
    index='Variant',
    columns='Model',
    values=['Recall', 'Precision', 'MRR', 'Accuracy', 'LeaderboardScore', 'DeltaVsBaseline'],
    observed=False,
)
phase2_best_by_model_df = pd.DataFrame(phase2_best_by_model).sort_values('Model').reset_index(drop=True)

display(phase2_summary_df)
display(phase2_cross_model_pivot)
display(phase2_best_by_model_df)



===== Phase 2 comparison for tfidf =====
Starting retrieval: model=tfidf
  parameters: top_k=12,500, docs=216,041, queries=327, prepared_artifacts=yes, embedding_kind='queries_train'
  [TF-IDF] vectorized 327 queries against 216,041 docs with capped_top_k=12,500


  [TF-IDF] processed 65/327 queries


  [TF-IDF] processed 130/327 queries


  [TF-IDF] processed 195/327 queries


  [TF-IDF] processed 260/327 queries


...

,Model,Variant,TopK,Recall,Precision,MRR,Accuracy,LeaderboardScore,DeltaVsBaseline
0,bm25,baseline,12500,0.717214,0.000449,0.282023,0.926606,0.481573,0.000000
1,bm25,category_filtered,12500,0.737898,0.000502,0.269339,0.926606,0.483586,0.002013
2,bm25,reranked,12500,0.717214,0.000449,0.467999,0.926606,0.528067,0.046494
3,bm25,reranked_plus_category_bonus,12500,0.717214,0.000449,0.468184,0.926606,0.528113,0.046540
4,bm25,category_filtered_plus_rerank,12500,0.737898,0.000502,0.429715,0.926606,0.523680,0.042107


,Model,Variant,TopK,Recall,Precision,MRR,Accuracy,LeaderboardScore,DeltaVsBaseline
0,embedding,baseline,12500,0.965746,0.000685,0.474507,0.926606,0.591886,0.000000
1,embedding,category_filtered,12500,0.905954,0.000657,0.442439,0.926606,0.568914,-0.022972
2,embedding,reranked,12500,0.965746,0.000685,0.643416,0.926606,0.634113,0.042227
3,embedding,reranked_plus_category_bonus,12500,0.965746,0.000685,0.648262,0.926606,0.635325,0.043439
4,embedding,category_filtered_plus_rerank,12500,0.905954,0.000657,0.601620,0.926606,0.608709,0.016823


,Model,Variant,TopK,Recall,Precision,MRR,Accuracy,LeaderboardScore,BaselineLeaderboardScore,DeltaVsBaseline
0,bm25,baseline,12500,0.717214,0.000449,0.282023,0.926606,0.481573,0.481573,0.000000
1,bm25,category_filtered,12500,0.737898,0.000502,0.269339,0.926606,0.483586,0.481573,0.002013
2,bm25,reranked,12500,0.717214,0.000449,0.467999,0.926606,0.528067,0.481573,0.046494
3,bm25,reranked_plus_category_bonus,12500,0.717214,0.000449,0.468184,0.926606,0.528113,0.481573,0.046540
4,bm25,category_filtered_plus_rerank,12500,0.737898,0.000502,0.429715,0.926606,0.523680,0.481573,0.042107
5,embedding,baseline,12500,0.965746,0.000685,0.474507,0.926606,0.591886,0.591886,0.000000
6,embedding,category_filtered,12500,0.905954,0.000657,0.442439,0.926606,0.568914,0.591886,-0.022972
7,embedding,reranked,12500,0.965746,0.000685,0.643416,0.926606,0.634113,0.591886,0.042227
8,embedding,reranked_plus_category_bonus,12500,0.965746,0.000685,0.648262,0.926606,0.635325,0.591886,0.043439
9,embedding,category_filtered_plus_rerank,12500,0.905954,0.000657,0.601620,0.926606,0.608709,0.591886,0.016823


Accuracy                     DeltaVsBaseline  \
Model                              bm25 embedding     tfidf            bm25   
Variant                                                                       
baseline                       0.926606  0.926606  0.926606        0.000000   
category_filtered              0.926606  0.926606  0.926606        0.002013   
reranked                       0.926606  0.926606  0.926606        0.046494   
reranked_plus_category_bonus   0.926606  0.926606  0.926606        0.046540   
category_filtered_plus_rerank  0.926606  0.926606  0.926606        0.042107   

                                                  LeaderboardScore            \
Model                         embedding     tfidf             bm25 embedding   
Variant                                                                        
baseline                       0.000000  0.000000         0.481573  0.591886   
category_filtered             -0.022972 -0.000983         0.483586  0.568914   
reranked                       0.042227  0.053947         0.528067  0.634113   
reranked_plus_category_bonus   0.043439  0.054415         0.528113  0.635325   
category_filtered_plus_rerank  0.016823  0.053742         0.523680  0.608709   

                                              MRR                      \
Model                             tfidf      bm25 embedding     tfidf   
Variant                                                                 
baseline                       0.494993  0.282023  0.474507  0.262707   
category_filtered              0.494010  0.269339  0.442439  0.253238   
reranked                       0.548940  0.467999  0.643416  0.478497   
reranked_plus_category_bonus   0.549408  0.468184  0.648262  0.480366   
category_filtered_plus_rerank  0.548735  0.429715  0.601620  0.472139   

                              Precision                        Recall  \
Model                              bm25 embedding     tfidf      bm25   
Variant                                                                 
baseline                       0.000449  0.000685  0.000490  0.717214   
category_filtered              0.000502  0.000657  0.000535  0.737898   
reranked                       0.000449  0.000685  0.000490  0.717214   
reranked_plus_category_bonus   0.000449  0.000685  0.000490  0.717214   
category_filtered_plus_rerank  0.000502  0.000657  0.000535  0.737898   

                                                   
Model                         embedding     tfidf  
Variant                                            
baseline                       0.965746  0.790170  
category_filtered              0.905954  0.795661  
reranked                       0.965746  0.790170  
reranked_plus_category_bonus   0.965746  0.790170  
category_filtered_plus_rerank  0.905954  0.795661

,Model,Variant,TopK,Recall,Precision,MRR,Accuracy,LeaderboardScore,DeltaVsBaseline
0,bm25,reranked_plus_category_bonus,12500,0.717214,0.000449,0.468184,0.926606,0.528113,0.046540
1,embedding,reranked_plus_category_bonus,12500,0.965746,0.000685,0.648262,0.926606,0.635325,0.043439
2,tfidf,reranked_plus_category_bonus,12500,0.790170,0.000490,0.480366,0.926606,0.549408,0.054415
